In [ ]:
!wget https://raw.githubusercontent.com/JoseCaliz/dotfiles/main/css/custom_css.css 2>/dev/null 1>&2
!pip install feature_engine 2>/dev/null 1>&2
    
from IPython.core.display import HTML
with open('./custom_css.css', 'r') as file:
    custom_css = file.read()

HTML(custom_css)

# TPS Sep 22

<img src='https://i.imgur.com/nDOJG1T.png' style='width:50%; margin-left:auto; margin-right:auto'/>

<span id='toc'/>

# Table of Content
1. [TPS 22](#TPS-22)
1. [Table of Content](#Table-of-Content)
1. [Library Import](#Library-Import)
1. [Read Data](#Read-Data)
1. [Visualize your Data and Draw Conclusions](#Visualize-your-Data-and-Draw-Conclusions)
    1. [product](#product)
    1. [Total sells by Country](#Total-sells-by-Country)
    1. [Date Range on Train and Test](#Date-Range-on-Train-and-Test)
    1. [Null Data](#Null-Data)
    1. [Sells by Product](#Sells-by-Product)
    1. [Sells by Product &amp; Store](#Sells-by-Product-&-Store)
    1. [Seasonality](#Seasonality)
        1. [Day](#Day)
        1. [Week Of The Year](#Week-Of-The-Year)
        1. [Month](#Month)
        1. [Day Of Week](#Day-Of-Week)
        1. [Quarter](#Quarter)
        1. [Day Of Year](#Day-Of-Year)
1. [Feature Egineering](#Feature-Egineering)
1. [Modeling](#Modeling)
    1. [Metrics](#Metrics)
    1. [Features Selected](#Features-Selected)
    1. [Multiplicative Model](#Multiplicative-Model)
1. [Submission](#Submission)

# Library Import

[Take me to the TOC 🌠](#toc)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.cm as cmap
import matplotlib.colors as mpl_colors
import matplotlib.pyplot as plt
import  matplotlib.ticker as ticker
import matplotlib as mpl
from itertools import product as cartessian_product
from feature_engine.encoding import OneHotEncoder
import dateutil.easter as easter
from rich import print
import numpy as np
from sklearn.linear_model import LinearRegression

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

cluster_colors_hex = ['#b4d2b1', '#568f8b', '#1d4a60', '#cd7e59', '#ddb247', '#d15252']
cluster_colors_rgb = [hex_to_rgb(x) for x in cluster_colors_hex]
cmap = mpl_colors.ListedColormap(cluster_colors_rgb)
colors = cmap.colors
bg_color= '#fdfcf6'

custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    'grid.alpha':0.3,
    'figure.figsize': (16, 6),
    'axes.titlesize': 'Large',
    'axes.labelsize': 'Large',
    'figure.facecolor': bg_color,
    'axes.facecolor': bg_color
}

sns.set_theme(
    style='whitegrid',
    palette=sns.color_palette(cluster_colors_hex),
    rc=custom_params
)

# Read Data

[Take me to the TOC 🌠](#toc)

In [ ]:
train_df = pd.read_csv(
    "../input/tabular-playground-series-sep-2022/train.csv",
    parse_dates=['date'],
    index_col=0
)

test_df = pd.read_csv(
    "../input/tabular-playground-series-sep-2022/test.csv",
    parse_dates=['date'],
    index_col=0
)

# Visualize your Data and Draw Conclusions

## product
[Take me to the TOC 🌠](#toc)

In [ ]:
ax = sns.barplot(data=train_df, x='product', y='num_sold', hue='country');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>All products are sold by all countries</li>
        <li>Products have different volumes, use product as a feature</li>
        <li>Belgium and Germany are the ones that sell more, use country as a feature</li>
    </ul>
</div>

## Total sells by Country
[Take me to the TOC 🌠](#toc)

In [ ]:
ax = sns.lineplot(
    data=train_df.groupby([
        train_df.date.dt.strftime('%Y-%m'),
        train_df.country
    ])['num_sold'].sum().reset_index(),
    x='date',
    y='num_sold',
    hue='country',
)


ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=20))

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Something happpened on early 2020... Pandemic effects?</li>
        <li>If using ML models, we should weight our instances by year</li>
    </ul>
</div>

## Date Range on Train and Test
[Take me to the TOC 🌠](#toc)

In [ ]:
train_range = train_df.date.agg(['min', 'max'])
test_range = test_df.date.agg(['min', 'max'])


fig, ax = plt.subplots()
ax.plot(train_range, [0.2, 0.2], linewidth=5, marker='.', markersize=30, label='train')
ax.plot(test_range, [0.3, 0.3], linewidth=5, marker='.', markersize=30, label='test', color=colors[1])
ax.set_ylim(0.1, 0.4)
ax.legend()
ax.set_yticks(ticks=[]);
plt.xticks(fontsize=14)
ax.legend(borderpad=1.5, fontsize=13)

print('train', train_range, '\n', sep='\n')
print('test', test_range, sep='\n')

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>No overlap between Train and test</li>
        <li>Year cannot be used as a feature</li>
    </ul>
</div>

## Null Data
[Take me to the TOC 🌠](#toc)

In [ ]:
fig = plt.figure(tight_layout=True, figsize=(20, 9))
gs = mpl.gridspec.GridSpec(2, 1)
null_table = []

# plot bars
for i, (k, df) in enumerate({'train':train_df, 'test':test_df}.items()):
    ax = fig.add_subplot(gs[i, 0])
    null_table.append((df.isnull().sum()/df.shape[0]).rename(k))
    
    sns.histplot(
        data=df.drop('Transported', errors='ignore', axis=1).isna().melt(value_name="missing"),
        y="variable",
        hue="missing",
        multiple="fill",
        ax=ax
    )

    ax.xaxis.set_major_locator(ticker.LinearLocator(21))
    ax.xaxis.set_major_formatter('{:.0%}'.format)
    ax.set_title(k, fontsize=15)
    ax.set_xlabel('Null Percentage');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>No Null variables, no need to impute</li>
    </ul>
</div>

## Distrubutions of num_solds

In [ ]:
fig, ax = plt.subplots(12, 4, figsize=(25, 50))
ax = ax.flatten()

for i, (combination, df) in enumerate(train_df.groupby(['country', 'store', 'product'])):
    sns.histplot(df.num_sold, ax=ax[i])
    ax[i].set_title(' | '.join(combination))
plt.tight_layout()

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Seriouly, what is wrong with Poland?</li>
        <li>Distributions are skewed in most cases. SMAPE penalizes more understimates than overstimates so it choosing a right loss is mandatory.</li>
    </ul>
</div>

## Year tendency

In [ ]:
fig, ax = plt.subplots(12, 4, figsize=(25, 50))
ax = ax.flatten()
for i, (combination, df) in enumerate(train_df.groupby(['country', 'store', 'product'])):
    data = df.groupby(df.date.dt.year)['num_sold'].sum().reset_index()
    sns.barplot(data=data, x='date', y='num_sold', ax=ax[i])
    ax[i].plot(
        ax[i].get_xticks(), data.num_sold,
        color=colors[5], linestyle='--', marker='.',
        markersize=20, linewidth=2
    )
    lims = ax[i].get_ylim()
    ax[i].set_ylim(data.num_sold.min()*0.7, lims[1])
    ax[i].set_title(' | '.join(combination))
plt.tight_layout()

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>This is probably subject to questionining but only France is stable on 2020</li>
    </ul>
</div>

## Sells by Product
[Take me to the TOC 🌠](#toc)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(18, 14), sharey=True)
ax = ax.flatten()
hue_order = train_df.country.unique()

for i, product in enumerate(train_df['product'].unique()):
    df = train_df[train_df['product'] == product]
    sns.lineplot(
        data=df.groupby([
            df.date.dt.strftime('%Y-%m'),
            df.country
        ])['num_sold'].sum().reset_index(),
        x='date',
        y='num_sold',
        hue='country',
        ax=ax[i],
        hue_order=hue_order
    )
    ax[i].set_title(product)
    ax[i].xaxis.set_major_locator(ticker.MaxNLocator(nbins=10))
    
    handles = ax[i].get_legend_handles_labels()[0]
    labels = ax[i].get_legend_handles_labels()[1]
    ax[i].legend().remove()
    
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), fontsize=14, ncol=6)
plt.tight_layout()

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>All products have that strange behavior on 2020</li>
        <li>Differences between countries are clear except for Belgium and France</li>
        <li>Clear month seasonality except on "Kaggle For Kids", probabily good idea to encode the month</li>
    </ul>
</div>

## Sells by Product & Store

[Take me to the TOC 🌠](#toc)


In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(18, 20), sharey=True)
ax = ax.flatten()
hue_order = train_df.country.unique()

iterator = cartessian_product(train_df['product'].unique(), train_df['store'].unique())
for i, (product, store) in enumerate(iterator):
    df_to_plot = train_df[
        train_df.store.eq(store) & 
        train_df['product'].eq(product)
    ]
    sns.lineplot(
        data=df_to_plot.groupby([
            df_to_plot.date.dt.strftime('%Y-%m'),
            df_to_plot.country
        ])['num_sold'].sum().reset_index(),
        x='date',
        y='num_sold',
        hue='country',
        ax=ax[i],
        hue_order=hue_order
    )
    ax[i].set_title(f'{store} | {product}')
    ax[i].xaxis.set_major_locator(ticker.MaxNLocator(nbins=10))
    
    handles = ax[i].get_legend_handles_labels()[0]
    labels = ax[i].get_legend_handles_labels()[1]
    ax[i].legend().remove()
    
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), fontsize=14, ncol=6)
plt.tight_layout()

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>KaggleRama has lower traffic than KaggleMart, store must be a feature</li>
        <li>Month seasonality remains</li>
    </ul>
</div>

## Seasonality

Inspired on [this notebook](https://www.kaggle.com/code/lucamassaron/kaggle-merchandise-eda-with-baseline-linear-model)

[Take me to the TOC 🌠](#toc)

In [ ]:
def process_time(df):
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['week'] = df['date'].dt.isocalendar().week
    df.loc[df['week'] > 52, 'week'] = 52
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['dayofyear'] = df['date'].dt.dayofyear

process_time(train_df)
process_time(test_df)

def plot_seasonality(season_col, ci=95, alpha=1):
    fig, ax = plt.subplots(4, 2, figsize=(18, 20))
    ax = ax.flatten()
    iterator = cartessian_product(train_df['product'].unique(), train_df['store'].unique())

    for i, (product, store) in enumerate(iterator):
        df_to_plot = train_df[
            train_df.store.eq(store) & 
            train_df['product'].eq(product)
        ]
        sns.lineplot(
            data=pd.melt(
                df_to_plot, id_vars=['country', season_col],
                value_vars=['num_sold'],
                value_name='num_solds'
            ),
            x=season_col,
            y='num_solds',
            hue='country',
            ax=ax[i],
            ci=ci,
            alpha=alpha,
            hue_order=hue_order
        )
        ax[i].set_title(f'{store} | {product}')

        handles = ax[i].get_legend_handles_labels()[0]
        labels = ax[i].get_legend_handles_labels()[1]
        ax[i].legend().remove()

    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), fontsize=14, ncol=6)
    plt.suptitle(f'Seasonality by {season_col}', fontsize=16)
    plt.tight_layout()
    return ax

### Day

[Take me to the TOC 🌠](#toc)

In [ ]:
plot_seasonality('day');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Variability increases by the end of the month. Worth adding a boolean variable <code>EndOfMonth</code></li>
        <li>Poland is more variable than other countries, needs further investigation</li>
    </ul>
</div>

### Week Of The Year

[Take me to the TOC 🌠](#toc)

In [ ]:
plot_seasonality('week');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Advanced Technique is more popular starting the year, GettingStarted is the other way around</li>
        <li>Kaggle for Kids is more popular by Xmas same as Kaggle Recipe </li>
    </ul>
</div>

### Month
[Take me to the TOC 🌠](#toc)

In [ ]:
plot_seasonality('month');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Same as week of the year but seasonality is more pronounced</li>
    </ul>
</div>

### Day Of Week

[Take me to the TOC 🌠](#toc)

In [ ]:
plot_seasonality('dayofweek');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Friday, Saturdays and Sundays are the days where more items are sold, we should create a boolean features <code>isFriday</code> and <code>isSaturday</code> and <code>isSunday</code> </li>
    </ul>
</div>

### Quarter
[Take me to the TOC 🌠](#toc)

In [ ]:
plot_seasonality('quarter');

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Same as month</li>
    </ul>
</div>

### Day Of Year

[Take me to the TOC 🌠](#toc)

In [ ]:
ax = plot_seasonality('dayofyear', ci=None)

for ax_i in ax:
    # 357 = 23 Dec
    ax_i.axvline(357, linestyle=':', color='purple', alpha=0.4)

<div class='insights'>
    <b> 🔑 Insights </b>
    <ul>
        <li>Hello end-of-year holidays effect, add boolean features if date is in between Dec24 and Jan1st</li>
    </ul>
</div>

# Feature Egineering

[Take me to the TOC 🌠](#toc)

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['friday'] = df.date.dt.weekday.eq(4)
    df['saturday'] = df.date.dt.weekday.eq(5)

    #Encode country, store & product
    encoder = OneHotEncoder(variables=['country', 'store', 'product'], drop_last=True)
    df = encoder.fit_transform(df)
    df['dayofyear_sin'] = np.sin(df.dayofyear / 365 * 2 * np.pi)
    df['dayofyear_cos'] = np.cos(df.dayofyear / 365 * 2 * np.pi)
    
    #X-mas Holidays Indicator
    for day in range(24, 32):
        df[f'dec_{day}'] = df.date.dt.day.eq(day) & df.date.dt.month.eq(12)
    
    # Eeaster Indicator
    ## Get dates surrounding easter and 40 days after
    easter_date = df.date.apply(lambda date: pd.Timestamp(easter.easter(date.year)))
    for day in list(range(-5, 5)) + list(range(40, 48)):
        df[f'easter_{day}'] = (df.date - easter_date).dt.days.eq(day)
    
    df.drop(columns=['date'], inplace=True)
    return df

train_test = engineer_features(train_df.append(test_df))
train_df_ = train_test.loc[train_df.index].copy()
test_df_ = train_test.iloc[test_df.index].copy()
test_df_.drop(columns=['num_sold'], inplace=True)

target = train_df_.pop('num_sold')

# Modeling

## Metrics

Taken from [https://www.kaggle.com/cpmpml/smape-weirdness](#https://www.kaggle.com/cpmpml/smape-weirdness)

[Take me to the TOC 🌠](#toc)

In [ ]:
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 200.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return np.nanmean(diff)

## Features Selected

[Take me to the TOC 🌠](#toc)

In [ ]:
print('features', train_df_.columns.tolist())

## Additive Model

[Take me to the TOC 🌠](#toc)

In [ ]:
linear_model = LinearRegression()
linear_model.fit(train_df_, target)
preds = linear_model.predict(train_df_)
print(f"SMAPE: {smape(y_true=target, y_pred=preds):.4f}")

# Submission

[Take me to the TOC 🌠](#toc)

In [ ]:
test_preds = linear_model.predict(test_df_)
submission = pd.DataFrame({'num_sold':test_preds}, index=test_df_.index)
submission.to_csv('submission.csv')